In [10]:
!pip install datasets pandas -q

In [11]:
from datasets import load_dataset
import pandas as pd
import json
import os

## Load dữ liệu

In [12]:
print("Đang tải dữ liệu banking77...")
raw_dataset = load_dataset("banking77")

print("\n--- CẤU TRÚC TẬP DỮ LIỆU ---")
print(dataset)

train_data = dataset['train']
test_data = dataset['test']
print(f"\nSố lượng mẫu tập Train: {len(train_data)}")
print(f"Số lượng mẫu tập Test: {len(test_data)}")

features = train_data.features
num_classes = features['label'].num_classes
class_names = features['label'].names
print(f"\nTổng số Intent (class): {num_classes}")
print(f"Ví dụ 5 intent đầu tiên: {class_names[:5]}")

print("\n--- 5 MẪU DỮ LIỆU ĐẦU TIÊN (TẬP TRAIN) ---")
df_sample = pd.DataFrame(train_data[:5])
df_sample['intent_name'] = df_sample['label'].apply(lambda x: class_names[x])
pd.set_option('display.max_colwidth', None)
display(df_sample[['text', 'label', 'intent_name']])

Đang tải dữ liệu banking77...

--- CẤU TRÚC TẬP DỮ LIỆU ---
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})

Số lượng mẫu tập Train: 10003
Số lượng mẫu tập Test: 3080

Tổng số Intent (class): 77
Ví dụ 5 intent đầu tiên: ['activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support', 'automatic_top_up']

--- 5 MẪU DỮ LIỆU ĐẦU TIÊN (TẬP TRAIN) ---


,text,label,intent_name
0,I am still waiting on my card?,11,card_arrival
1,What can I do if my card still hasn't arrived after 2 weeks?,11,card_arrival
2,I have been waiting over a week. Is the card still coming?,11,card_arrival
3,Can I track my card while it is in the process of delivery?,11,card_arrival
4,"How do I know if I will get my card, or if it is lost?",11,card_arrival


## Label Mapping

In [13]:
id2label = {i: name for i, name in enumerate(class_names)}
label2id = {name: i for i, name in enumerate(class_names)}

config_path = "configs/label_mapping.json"
os.makedirs(os.path.dirname(config_path), exist_ok=True)

mapping_data = {
    "id2label": id2label,
    "label2id": label2id
}

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(mapping_data, f, indent=4, ensure_ascii=False)

print(f"Đã lưu ánh xạ nhãn vào {config_path}")

Đã lưu ánh xạ nhãn vào configs/label_mapping.json


## Chia dữ liệu

In [14]:
df_all = pd.concat([pd.DataFrame(raw_dataset['train']), pd.DataFrame(raw_dataset['test'])], ignore_index=True)
# Lấy 50 mẫu cho mỗi intent
SAMPLES_PER_INTENT = 50
subset_df = df_all.groupby('label').apply(
    lambda x: x.sample(n=SAMPLES_PER_INTENT, random_state=42)
).reset_index(drop=True)
print(f"Hoàn thành! Đã lấy {len(subset_df)} mẫu ({SAMPLES_PER_INTENT} mẫu cho mỗi nhãn trong 77 nhãn).")

Hoàn thành! Đã lấy 3850 mẫu (50 mẫu cho mỗi nhãn trong 77 nhãn).


/tmp/ipykernel_57/2807374927.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  subset_df = df_all.groupby('label').apply(


In [15]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    subset_df, 
    test_size=0.2, 
    stratify=subset_df['label'], 
    random_state=42
)

os.makedirs("sample_data", exist_ok=True)
train_df.to_csv("sample_data/train.csv", index=False)
test_df.to_csv("sample_data/test.csv", index=False)

print(f"Đã lưu dữ liệu vào thư mục 'sample_data/':")
print(f"- Train: {len(train_df)} mẫu (khoảng 40 mẫu/intent)")
print(f"- Test: {len(test_df)} mẫu (khoảng 10 mẫu/intent)")

# Hiển thị kiểm tra 5 mẫu đầu tiên
display(train_df.head())

Đã lưu dữ liệu vào thư mục 'sample_data/':
- Train: 3080 mẫu (khoảng 40 mẫu/intent)
- Test: 770 mẫu (khoảng 10 mẫu/intent)


,text,label
3632,Why was my virtual card rejected?,72
1077,Can I change my pin at a cash machine?,21
2714,Can I electronically transfer funds into my account from my American Express?,54
3476,How and where can I find the identity checks?,69
1578,Is there a fee to exchange foreign money?,31
